# 3. 인공지능기본법 Agentic RAG

- 참고 실습: `track1_core/03-3_agentic_rag_basic.ipynb`
- 지식원: 인공지능기본법 HWPX의 조문·항·호·목 Chunk


## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. 일반 RAG와 Agentic RAG의 실행 구조 차이를 설명할 수 있다.
2. [03-3A] 검색 필요성 판단, 검색, 답변 생성의 기본 Graph를 완성할 수 있다.
3. [03-3B] 기본 Graph에 Query Rewrite와 근거 적합성 평가를 추가할 수 있다.
4. [03-3B] 재검색 조건과 최대 재검색 횟수를 판단하는 조건 분기를 구현할 수 있다.
5. 근거가 부족할 때 LLM의 일반지식만으로 확정적인 답변을 만들지 않고, 근거 부족을
   명시적으로 알리는 응답을 구현할 수 있다.
6. Agentic RAG가 일반 RAG보다 항상 우수한 것은 아니며, 복잡성과 호출 비용이
   늘어난다는 점을 관찰을 통해 설명할 수 있다.


## 2. 문제 상황

법률 질의는 반드시 확인된 조문을 근거로 답해야 한다. Agentic RAG는 법령 검색이
필요한 질문인지 판단하고, 조문 검색 결과가 질문에 충분한지 평가하며, 근거가
부족하면 검색어를 수정한다. 답변 인용은 모델이 임의로 만들지 않고 실제 검색
metadata의 법률명·장·조문에서 구성한다.


## 3. 핵심 개념

### 3.1 정의

일반 RAG는 질문이 들어오면 항상 검색을 실행하고, 검색된 문서를 그대로 근거로
삼아 답변을 생성하는 고정된 파이프라인이다. Agentic RAG는 검색이 필요한지,
검색 결과가 충분한 근거가 되는지를 판단하고, 필요하면 검색어를 바꿔 재검색하는
등 스스로 판단하며 흐름을 조절하는 RAG다.

### 3.2 개념이 필요한 이유

모든 질문에 검색이 필요한 것은 아니고("1 더하기 1은 얼마인가요?"), 모든 검색
결과가 질문에 답하기 충분한 것도 아니다. 이 판단을 Workflow에 내장하지 않으면,
불필요한 검색을 하거나 근거 없는 확정적 답변을 만들어내는 문제가 생긴다.

### 3.3 주요 구성요소

`RAGState`의 각 필드는 다음 역할을 한다.

| 필드 | 역할 |
|---|---|
| `question` | 사용자의 원본 질문 |
| `search_required` | 검색이 필요한지 여부 |
| `search_queries` | 지금까지 시도한 검색어 목록(재검색 시 누적) |
| `retrieved_documents` | 가장 최근 검색으로 얻은 문서 목록 |
| `evidence_valid` | 검색 결과가 충분한 근거인지 여부 |
| `retry_count` | 근거 부족으로 재검색을 시도한 횟수 |
| `citations` | 최종 답변이 근거로 삼은 문서 출처 목록 |
| `final_answer` | 최종적으로 사용자에게 전달할 답변 |

**일반 RAG와 Agentic RAG 비교**

| 구분 | 일반 RAG | Agentic RAG |
|---|---|---|
| 검색 실행 | 항상 또는 고정 | 필요성 판단 가능 |
| 검색어 | 사용자 질문 중심 | 생성·수정 가능 |
| 검색 결과 평가 | 선택적 | 근거 적합성 평가 |
| 재검색 | 일반적으로 고정되지 않음 | 조건에 따라 수행 |
| 종료 조건 | 단순 | 명시적 조건 필요 |

### 3.4 동작 과정

**필수 Workflow**

```text
질문
  ↓
검색 필요성 판단
  ├─ 검색 불필요 → 직접 응답
  └─ 검색 필요
        ↓
     검색어 생성
        ↓
     문서 검색
        ↓
     근거 적합성 평가
        ├─ 적합 → 답변 생성
        └─ 부적합 → 검색어 수정(재검색, 최대 횟수까지)
                     └─ 최대 횟수 초과 → 근거 부족 응답
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `decide_search_node` | 검색 필요성 판단 |
| `decide_search_required()` | 검색 필요성 판단 로직(LLM 호출) |
| `generate_query_node` | 검색어 생성(Query Rewrite) |
| `retrieve_node` | 문서 검색 |
| `evaluate_evidence_node` | 근거 적합성 평가 |
| `route_after_evidence()` | 재검색 조건과 최대 재검색 횟수 판단 |
| `generate_answer_node` | 근거 기반 답변 생성과 출처 표시 |
| `insufficient_evidence_node` | 근거 부족 응답 |

### 3.6 유사 개념과의 차이

일반 RAG는 "검색 → 답변 생성"이 고정된 2단계 파이프라인이지만, Agentic RAG는
그 사이에 "판단"이 여러 번 끼어든다. 겉보기에 Agentic RAG가 항상 더 정확해
보일 수 있지만, 실제로는 판단이 틀리면(예: 근거가 충분한데 부족하다고 잘못
평가) 불필요한 재검색만 반복하며 비용과 지연시간만 늘어날 수 있다.

### 3.7 사용 시점과 적용 조건

질문의 난이도가 다양하고, 문서에 답이 없을 수도 있는 상황(사용자 지원, 사내
FAQ 등)에서는 근거 적합성 평가와 근거 부족 응답이 중요하다. 반대로 검색 대상
문서가 항상 신뢰할 수 있고 질문 유형이 단순하다면, 일반 RAG로도 충분할 수
있다.

### 3.8 한계와 주의사항

- 검색 필요성 판단과 근거 적합성 평가 자체가 LLM 호출이므로, 판단이 틀릴 수
  있다.
- 재검색을 반복할수록 LLM 호출 횟수와 지연시간이 늘어난다.
- 최대 재검색 횟수를 넉넉하게 두지 않으면, 실제로는 검색어를 조금만 바꾸면
  찾을 수 있는 근거도 놓칠 수 있다.

### 3.9 자주 발생하는 오해

"Agentic RAG는 항상 일반 RAG보다 정확하고 우수하다"는 오해가 흔하다. 실제로는
Agentic RAG가 복잡성, 호출 비용, 지연시간을 늘리는 대가로 근거 검증이라는
안전장치를 얻는 것이며, 문제 상황에 따라 일반 RAG로 충분한 경우도 많다.
**검색 결과가 없는데 LLM의 일반지식만으로 확정적인 답변을 생성해서는 안
된다**는 점도 반드시 지켜야 하는 설계 원칙이다.

### 3.10 핵심 정리

- Agentic RAG는 검색 필요성 판단, 검색어 생성, 근거 적합성 평가, 재검색을
  Workflow에 내장한 RAG다.
- `RAGState`는 판단 결과(`search_required`, `evidence_valid`)와 진행 상태
  (`search_queries`, `retry_count`)를 함께 관리한다.
- 재검색은 근거가 부족하고 최대 횟수 이내일 때만 수행되며, 초과하면 근거 부족을
  명시적으로 알린다.
- Agentic RAG는 항상 우수한 것이 아니라, 복잡성과 비용을 감수하고 근거 검증을
  얻는 설계 선택이다.


## 4. 실행 구조

### A) 기본 완성

```text
START → 검색 필요 판단 ── 불필요 → 직접 답변 → END
                    └─ 필요 → 검색어 생성 → 검색 → 답변 → END
```

이 지점에서 먼저 실행 가능한 `basic_rag_app`을 완성한다.

### B) Agentic 확장

```text
START
  ↓
decide_search ──(불필요)──▶ direct_answer ──▶ END
  │(필요)
  ▼
generate_query ──▶ retrieve ──▶ evaluate_evidence
                                    │
                    ┌───────────────┼───────────────┐
                 (적합)          (부적합,           (부적합,
                    │          retry_count<MAX)   retry_count>=MAX)
                    ▼                │                 │
             generate_answer         │                 ▼
                    │                │        insufficient_evidence
                    ▼                │                 │
                   END       (generate_query로 되돌아감)  ▼
                                                        END
```


## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [ ]:
import re
import zipfile
from xml.etree import ElementTree as ET

from typing import TypedDict

from langchain_core.documents import Document
from langgraph.graph import END, START, StateGraph

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_chat_model, get_embedding_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import CHROMA_DIR, OUTPUT_DIR, PROJECT_ROOT, data_path
from agentic_ai.retrieval_utils import add_documents_if_empty, get_chroma_store

settings = get_settings()
print_environment_summary(
    settings,
    needs_chat_model=True,
    needs_embedding_model=True,
)

chat_model = get_chat_model()
embedding_model = get_embedding_model()

LLM_CALL_COUNT = 0


def call_llm(prompt: str) -> str:
    # LLM 호출 횟수를 세면서 chat_model을 호출한다.
    global LLM_CALL_COUNT
    LLM_CALL_COUNT += 1
    return chat_model.invoke(prompt).content


## 6. 최소 실행 예제

전체 Graph를 만들기 전에, "검색이 필요한가?"를 LLM에게 묻는 프롬프트 하나만
먼저 실행해본다.


In [ ]:
mini_prompt = (
    "질문: 1 더하기 1은 얼마인가요?\n"
    "이 질문에 답하기 위해 별도의 문서 검색이 필요한지 판단하세요. "
    "SEARCH: yes 또는 SEARCH: no 형식으로만 답하세요."
)
mini_response = call_llm(mini_prompt)
print(f"LLM 응답: {mini_response}")
print(f"검색 필요 여부: {'SEARCH: yes' in mini_response}")


## 7. 03-3A 기본 RAG Graph 구현 [중급]

### 7.1 State 정의

작업지시서에서 요구하는 `RAGState`를 그대로 정의한다.


In [ ]:
# Graph의 모든 Node가 공유하고 부분적으로 갱신할 데이터 계약이다.
class RAGState(TypedDict):
    question: str
    search_required: bool
    search_queries: list[str]
    retrieved_documents: list[dict]
    evidence_valid: bool
    retry_count: int
    citations: list[dict]
    final_answer: str


MAX_RETRY = 2


### 7.2 Vector Store와 검색 함수 (03-2 구현 방식 활용)

문서 Embedding 저장, 유사도 계산, 정렬, Top-k 선택은 Vector Store에 맡긴다.
03-2의 Chroma 구성 방식을 활용하되 Agentic RAG 실험용 Collection을 별도로 만든다.
이 Notebook에서는 검색 결과를 Agent의 State에 연결하고
근거가 부족할 때 검색어를 바꿔 재검색하는 흐름에 집중한다. Chroma DB는
`outputs/vectorstore/chroma/`에 저장하고, 03-2 데이터와 섞이지 않도록 별도 Collection을
사용한다.


In [ ]:
DATA_PATH = data_path(
    "samples",
    "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).hwpx",
    must_exist=True,
)
LAW_ID = "ai-trust-basic-act-20676"
LAW_NAME = "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법"
SOURCE_FILE = DATA_PATH.name
COLLECTION_NAME = "aitrust_law_chunks_v1"

HP_NS = "http://www.hancom.co.kr/hwpml/2011/paragraph"
XML_NS = {"hp": HP_NS}


def load_hwpx_paragraphs(file_path) -> list[str]:
    """HWPX의 section XML에서 화면에 표시되는 문단 텍스트를 순서대로 추출한다."""
    paragraphs: list[str] = []
    with zipfile.ZipFile(file_path) as archive:
        section_names = sorted(
            name
            for name in archive.namelist()
            if re.fullmatch(r"Contents/section\d+\.xml", name)
        )
        if not section_names:
            raise ValueError(f"HWPX 본문 section XML을 찾을 수 없습니다: {file_path}")

        for section_name in section_names:
            root = ET.fromstring(archive.read(section_name))
            for paragraph in root.findall(".//hp:p", XML_NS):
                raw_text = "".join(
                    node.text or "" for node in paragraph.findall(".//hp:t", XML_NS)
                )
                normalized = re.sub(r"\s+", " ", raw_text).strip()
                if normalized:
                    paragraphs.append(normalized)
    return paragraphs


RAW_PARAGRAPHS = load_hwpx_paragraphs(DATA_PATH)
RAW_TEXT = "\n".join(RAW_PARAGRAPHS)
effective_date_match = re.search(r"\[시행\s+([^\]]+)\]", RAW_TEXT)
LAW_EFFECTIVE_DATE = effective_date_match.group(1) if effective_date_match else "확인 필요"

first_chapter_index = next(
    index for index, paragraph in enumerate(RAW_PARAGRAPHS)
    if re.match(r"^제\d+장\s+", paragraph)
)
LAW_BODY_PARAGRAPHS = RAW_PARAGRAPHS[first_chapter_index:]
document_text = "\n".join(LAW_BODY_PARAGRAPHS)

print(f"입력 파일: {DATA_PATH}")
print(f"전체 HWPX 문단: {len(RAW_PARAGRAPHS)}개")
print(f"법령 본문 문단: {len(LAW_BODY_PARAGRAPHS)}개 / {len(document_text):,}자")
print(f"시행일: {LAW_EFFECTIVE_DATE}")

CHAPTER_RE = re.compile(r"^제(?P<number>\d+)장\s+(?P<title>.+)$")
ADDENDUM_RE = re.compile(r"^부칙(?:\s|<|$)")
ARTICLE_RE = re.compile(
    r"^제(?P<number>\d+)조(?:의(?P<sub_number>\d+))?"
    r"\((?P<title>[^)]+)\)\s*(?P<body>.*)$"
)
PARAGRAPH_RE = re.compile(r"^[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]")
ITEM_RE = re.compile(r"^\d+(?:의\d+)?\.\s*")
SUBITEM_RE = re.compile(r"^[가-하]\.\s*")


def parse_law_articles(paragraphs: list[str]) -> list[dict]:
    """장·조문·부칙 경계를 인식해 법률을 조문 레코드로 변환한다."""
    articles: list[dict] = []
    chapter = ""
    scope = "본칙"
    current: dict | None = None

    def flush_current() -> None:
        nonlocal current
        if current is None:
            return
        current["text"] = "\n".join(current["lines"])
        current["length"] = len(current["text"])
        articles.append(current)
        current = None

    for paragraph in paragraphs:
        chapter_match = CHAPTER_RE.match(paragraph)
        if chapter_match:
            flush_current()
            scope = "본칙"
            chapter = paragraph
            continue

        if ADDENDUM_RE.match(paragraph):
            flush_current()
            scope = "부칙"
            chapter = paragraph
            continue

        article_match = ARTICLE_RE.match(paragraph)
        if article_match:
            flush_current()
            number = article_match.group("number")
            sub_number = article_match.group("sub_number") or ""
            article_label = f"제{number}조" + (f"의{sub_number}" if sub_number else "")
            scope_key = "main" if scope == "본칙" else "addendum"
            article_key = number + (f"-{sub_number}" if sub_number else "")
            article_id = f"{LAW_ID}-{scope_key}-article-{article_key}"
            title = article_match.group("title").strip()
            current = {
                "article_id": article_id,
                "document_id": LAW_ID,
                "law_name": LAW_NAME,
                "scope": scope,
                "chapter": chapter,
                "article": article_label,
                "article_number": article_key,
                "article_title": title,
                "section": f"{article_label}({title})",
                "source_file": SOURCE_FILE,
                "effective_date": LAW_EFFECTIVE_DATE,
                "lines": [paragraph],
            }
        elif current is not None:
            current["lines"].append(paragraph)

    flush_current()
    return articles


ARTICLES = parse_law_articles(LAW_BODY_PARAGRAPHS)
MAIN_ARTICLES = [article for article in ARTICLES if article["scope"] == "본칙"]
ADDENDUM_ARTICLES = [article for article in ARTICLES if article["scope"] == "부칙"]

print(f"조문 수: {len(ARTICLES)}개 (본칙 {len(MAIN_ARTICLES)}개 / 부칙 {len(ADDENDUM_ARTICLES)}개)")
print("첫 조문:", MAIN_ARTICLES[0]["section"])
print("마지막 본칙 조문:", MAIN_ARTICLES[-1]["section"])

def _body_lines(article: dict) -> list[str]:
    """첫 줄의 조문 표제는 제거하고 조문 본문만 반환한다."""
    first_match = ARTICLE_RE.match(article["lines"][0])
    first_body = first_match.group("body").strip() if first_match else article["lines"][0]
    return ([first_body] if first_body else []) + article["lines"][1:]


def _split_groups(lines: list[str], marker: re.Pattern) -> tuple[list[str], list[list[str]]]:
    """표지 문맥과 marker로 시작하는 연속 그룹을 분리한다."""
    prelude: list[str] = []
    groups: list[list[str]] = []
    current: list[str] | None = None
    for line in lines:
        if marker.match(line):
            if current:
                groups.append(current)
            current = [line]
        elif current is None:
            prelude.append(line)
        else:
            current.append(line)
    if current:
        groups.append(current)
    return prelude, groups


def _semantic_units(article: dict, budget: int) -> list[tuple[str, list[str]]]:
    """긴 조문을 항→호→목 경계로만 내려가며 의미 단위로 분리한다."""
    body = _body_lines(article)
    if not body:
        return [("article", [])]

    if any(PARAGRAPH_RE.match(line) for line in body):
        primary_type, primary_marker, secondary_type, secondary_marker = (
            "paragraph", PARAGRAPH_RE, "item", ITEM_RE
        )
    elif any(ITEM_RE.match(line) for line in body):
        primary_type, primary_marker, secondary_type, secondary_marker = (
            "item", ITEM_RE, "subitem", SUBITEM_RE
        )
    else:
        return [("article_part", body)]

    shared_intro, primary_groups = _split_groups(body, primary_marker)
    units: list[tuple[str, list[str]]] = []
    for primary_group in primary_groups:
        candidate = shared_intro + primary_group
        if len("\n".join(candidate)) <= budget:
            units.append((primary_type, candidate))
            continue

        local_intro, secondary_groups = _split_groups(primary_group, secondary_marker)
        if not secondary_groups:
            for line in primary_group:
                units.append((primary_type, shared_intro + [line]))
            continue

        for secondary_group in secondary_groups:
            secondary_candidate = shared_intro + local_intro + secondary_group
            if len("\n".join(secondary_candidate)) <= budget:
                units.append((secondary_type, secondary_candidate))
                continue

            tertiary_intro, tertiary_groups = _split_groups(secondary_group, SUBITEM_RE)
            if tertiary_groups:
                for tertiary_group in tertiary_groups:
                    units.append(("subitem", shared_intro + local_intro + tertiary_intro + tertiary_group))
            else:
                for line in secondary_group:
                    units.append((secondary_type, shared_intro + local_intro + [line]))
    return units or [("article_part", body)]


def _base_metadata(article: dict) -> dict:
    return {
        "document_id": article["document_id"],
        "law_name": article["law_name"],
        "scope": article["scope"],
        "chapter": article["chapter"],
        "article": article["article"],
        "article_number": article["article_number"],
        "article_title": article["article_title"],
        "section": article["section"],
        "source_file": article["source_file"],
        "effective_date": article["effective_date"],
        "hierarchy_path": f"{article['scope']} > {article['chapter']} > {article['section']}",
    }


def _context_prefix(article: dict) -> str:
    return f"{LAW_NAME}\n{article['scope']} | {article['chapter']}\n{article['section']}"


def build_parent_documents(articles: list[dict]) -> list[dict]:
    """답변 시 전체 조문 맥락으로 확장할 Parent 문서를 만든다."""
    parents = []
    for article in articles:
        text = f"{LAW_NAME}\n{article['scope']} | {article['chapter']}\n{article['text']}"
        parents.append({
            "doc_id": article["article_id"],
            **_base_metadata(article),
            "chunk_type": "article_parent",
            "parent_id": "",
            "text": text,
            "length": len(text),
        })
    return parents


def build_retrieval_documents(articles: list[dict], max_chars: int = 1000) -> list[dict]:
    """짧은 조문은 그대로, 긴 조문은 법률 계층 경계로 분할한다."""
    retrieval_documents: list[dict] = []
    for article in articles:
        prefix = _context_prefix(article)
        body_text = "\n".join(_body_lines(article))
        contextual_full_text = prefix + (f"\n{body_text}" if body_text else "")
        if len(contextual_full_text) <= max_chars:
            retrieval_documents.append({
                "doc_id": article["article_id"],
                **_base_metadata(article),
                "chunk_type": "article",
                "parent_id": "",
                "text": contextual_full_text,
                "length": len(contextual_full_text),
            })
            continue

        budget = max(200, max_chars - len(prefix) - 1)
        units = _semantic_units(article, budget)
        for index, (chunk_type, unit_lines) in enumerate(units, start=1):
            text = prefix + ("\n" + "\n".join(unit_lines) if unit_lines else "")
            retrieval_documents.append({
                "doc_id": f"{article['article_id']}-part-{index:02d}",
                **_base_metadata(article),
                "chunk_type": chunk_type,
                "parent_id": article["article_id"],
                "text": text,
                "length": len(text),
            })
    return retrieval_documents


PARENT_DOCUMENTS = build_parent_documents(ARTICLES)
RETRIEVAL_DOCUMENTS = build_retrieval_documents(ARTICLES, max_chars=1000)
SAMPLE_DOCUMENTS = RETRIEVAL_DOCUMENTS
PARENT_BY_ID = {document["doc_id"]: document for document in PARENT_DOCUMENTS}

long_article_children = [document for document in RETRIEVAL_DOCUMENTS if document["parent_id"]]
print(f"Parent 조문: {len(PARENT_DOCUMENTS)}개")
print(f"검색 Chunk: {len(RETRIEVAL_DOCUMENTS)}개 / 장문 조문 Child: {len(long_article_children)}개")
print("Chunk 유형:", sorted({document["chunk_type"] for document in RETRIEVAL_DOCUMENTS}))

# 본문은 임베딩 검색 대상으로, 식별자와 섹션은 인용용 metadata로 분리한다.
VECTOR_DOCUMENTS = [
    Document(
        page_content=record["text"],
        metadata={key: value for key, value in record.items() if key != "text"},
    )
    for record in SAMPLE_DOCUMENTS
]

vector_store = get_chroma_store(
    COLLECTION_NAME,
    embedding_model=embedding_model,
)
# 기존 Collection이 비어 있을 때만 적재해 재실행에 따른 문서 중복을 막는다.
documents_added, document_count = add_documents_if_empty(
    vector_store,
    VECTOR_DOCUMENTS,
    ids=[record["doc_id"] for record in SAMPLE_DOCUMENTS],
)

action = "초기화" if documents_added else "기존 Collection 재사용"
print(f"Chroma Collection: {COLLECTION_NAME}")
print(f"디스크 저장 경로: {CHROMA_DIR}")
print(f"처리 결과: {action} ({document_count}개 문서)")


def retrieve(query: str, k: int = 4) -> list[dict]:
    # Vector Store에서 질문과 유사한 상위 k개 문서를 가져온다.
    if k < 1:
        raise ValueError("k는 1 이상이어야 합니다.")
    # 유사도는 후보 순위를 나타낼 뿐 질문에 대한 충분한 근거인지는 다음 Node가 판단한다.
    matches = vector_store.similarity_search_with_relevance_scores(query, k=k)
    return [
        {**document.metadata, "text": document.page_content, "similarity": score}
        for document, score in matches
    ]


### 7.3 검색 필요성 판단

**TODO**: `decide_search_required()`를 작성한다.

- 질문을 포함한 프롬프트를 만들어 `call_llm()`으로 LLM에게 검색 필요 여부를
  묻는다.
- 프롬프트는 반드시 `"SEARCH: yes 또는 SEARCH: no 형식으로만 답하세요."`
  문구를 포함해야 한다(채점/모킹 로직이 이 문구를 기준으로 응답을 해석한다).
- 응답에 `"SEARCH: yes"`가 포함되어 있으면 `True`, 아니면 `False`를 반환한다.


In [ ]:
INTERNAL_KNOWLEDGE_HINTS = (
    "법률", "법령", "조문", "본칙", "부칙", "인공지능기본법", "고영향 인공지능",
    "생성형 인공지능", "인공지능사업자", "영향평가", "표시 의무", "과태료",
)


def decide_search_required(question: str) -> bool:
    if any(hint in question for hint in INTERNAL_KNOWLEDGE_HINTS):
        return True
    prompt = (
        f"질문: {question}\n"
        "법률 조문이나 내부 문서 근거가 필요하면 SEARCH: yes, "
        "일반 상식이나 단순 계산이면 SEARCH: no로만 답하세요."
    )
    response = call_llm(prompt).strip().lower()
    return response == "search: yes"


assert decide_search_required("1 더하기 1은 얼마인가요?") is False
assert decide_search_required("생성형 인공지능 표시 의무는 무엇인가요?") is True
print("decide_search_required 동작 확인 완료")


### 7.4 최초 검색어 생성 Node

03-3A에서는 원본 질문으로 최초 검색어를 만든다. 같은 Node는 03-3B에서 이전 검색어와
재검색 횟수를 함께 받아 Query Rewrite 역할로 확장된다.


In [ ]:
def generate_query_node(state: RAGState) -> dict:
    prompt = (
        f"원본 질문: {state['question']}\n"
        f"지금까지 시도한 검색어: {state['search_queries']}\n"
        f"재검색 시도 횟수: {state['retry_count']}\n"
        "검색어를 한 문장으로 작성하세요."
    )
    new_query = call_llm(prompt).strip()
    # 기존 검색어를 덮어쓰지 않고 누적해 재검색 과정 전체를 State에 보존한다.
    return {"search_queries": state["search_queries"] + [new_query]}


### 7.5 문서 검색 Node


In [ ]:
def retrieve_node(state: RAGState) -> dict:
    # 재검색이 발생하면 가장 최근에 생성된 검색어만 이번 검색에 사용한다.
    latest_query = state["search_queries"][-1]
    results = retrieve(latest_query, k=2)
    return {"retrieved_documents": results}


### 7.6 03-3A 기본 Graph 완성 지점

03-3A에서는 검색 결과를 별도로 평가하거나 재검색하지 않는다. 검색이 필요하면
최초 검색을 한 번 수행하고 바로 답변한다. 여기까지 정상 실행한 뒤에만 03-3B로
진행한다.

```text
검색 필요 판단 → 최초 검색어 → 검색 → 기본 답변
```


In [ ]:
def decide_search_node(state: RAGState) -> dict:
    return {"search_required": decide_search_required(state["question"])}


def route_after_search_decision(state: RAGState) -> str:
    # 반환 문자열은 conditional edge의 path map 키와 정확히 일치해야 한다.
    return "generate_query" if state["search_required"] else "direct_answer"


def direct_answer_node(state: RAGState) -> dict:
    answer = call_llm(f"질문: {state['question']}\n문서 검색 없이 직접 답변하세요.")
    return {"final_answer": answer, "citations": []}


def basic_answer_node(state: RAGState) -> dict:
    """03-3A: 검색 결과를 별도 평가하지 않고 한 번의 검색으로 답한다."""
    docs_text = "\n".join(f"[{d['doc_id']}] {d['text']}" for d in state["retrieved_documents"])
    answer = call_llm(
        f"질문: {state['question']}\n검색 문서를 참고해 답변하세요.\n{docs_text}"
    )
    # 인용은 모델이 생성하게 하지 않고 실제 검색된 문서 metadata에서 구성한다.
    citations = [
        {"doc_id": d["doc_id"], "section": d["section"]}
        for d in state["retrieved_documents"]
    ]
    return {"final_answer": answer, "citations": citations}


def initial_rag_state(question: str) -> RAGState:
    # 실행마다 새 리스트를 만들어 이전 질문의 검색 기록이나 인용이 섞이지 않게 한다.
    return {
        "question": question,
        "search_required": False,
        "search_queries": [],
        "retrieved_documents": [],
        "evidence_valid": False,
        "retry_count": 0,
        "citations": [],
        "final_answer": "",
    }


basic_graph = StateGraph(RAGState)
basic_graph.add_node("decide_search", decide_search_node)
basic_graph.add_node("direct_answer", direct_answer_node)
basic_graph.add_node("generate_query", generate_query_node)
basic_graph.add_node("retrieve", retrieve_node)
basic_graph.add_node("basic_answer", basic_answer_node)
basic_graph.add_edge(START, "decide_search")
basic_graph.add_conditional_edges(
    "decide_search",
    route_after_search_decision,
    {"direct_answer": "direct_answer", "generate_query": "generate_query"},
)
basic_graph.add_edge("direct_answer", END)
basic_graph.add_edge("generate_query", "retrieve")
basic_graph.add_edge("retrieve", "basic_answer")
basic_graph.add_edge("basic_answer", END)
basic_rag_app = basic_graph.compile()


def run_basic_rag(question: str) -> dict:
    return basic_rag_app.invoke(initial_rag_state(question))


basic_no_search = run_basic_rag("1 더하기 1은 얼마인가요?")
basic_search = run_basic_rag("생성형 인공지능 결과물 표시 의무는 무엇인가요?")
print("[A 직접 답변]", basic_no_search["final_answer"])
print("[A 검색 답변]", basic_search["final_answer"])
print("[A 인용]", basic_search["citations"])


## 8. B Agentic 확장 [고급]

> **A 완료 체크**: `basic_rag_app`에서 직접 답변 경로와 검색 답변 경로를 모두
> 실행한 뒤 진행한다. 03-3B는 기본 Graph를 대체하는 것이 아니라 판단과 Loop를
> 추가하는 확장 단계다.

### 8.1 근거 적합성 평가 Node

검색된 문서가 질문에 답하기 충분한 근거인지 LLM에게 판단을 맡긴다. 근거가
부족하다고 판단되면 `retry_count`를 1 늘린다.


In [ ]:
def evaluate_evidence_node(state: RAGState) -> dict:
    docs_text = "\n".join(f"- {d['text']}" for d in state["retrieved_documents"])
    prompt = (
        f"질문: {state['question']}\n"
        f"재검색 시도 횟수: {state['retry_count']}\n"
        f"검색된 문서:\n{docs_text}\n"
        "위 문서가 질문에 충분한 근거가 되는지 판단하세요. "
        "EVIDENCE: sufficient 또는 EVIDENCE: insufficient 형식으로만 답하세요."
    )
    response = call_llm(prompt)
    # 'insufficient' 안에도 'sufficient'가 포함되므로 부정 표현을 함께 배제한다.
    valid = "EVIDENCE: sufficient" in response and "insufficient" not in response
    update: dict = {"evidence_valid": valid}
    if not valid:
        # 실패한 평가에서만 횟수를 올려 MAX_RETRY가 실제 재검색 횟수를 제한하게 한다.
        update["retry_count"] = state["retry_count"] + 1
    return update


### 8.2 재검색 조건과 최대 재검색 횟수

**TODO**: `route_after_evidence()`를 작성한다. `evaluate_evidence_node` 다음에
실행할 Node 이름을 문자열로 반환하는 LangGraph 조건 분기 함수다.

- `state["evidence_valid"]`가 `True`이면 `"generate_answer"`를 반환한다.
- `evidence_valid`가 `False`이고 `state["retry_count"] < MAX_RETRY`이면
  `"retry"`를 반환한다.
- 그 외(`evidence_valid`가 `False`이고 재검색 횟수가 최대치에 도달)에는
  `"insufficient_evidence"`를 반환한다.


In [ ]:
def route_after_evidence(state: RAGState) -> str:
    """근거 적합성과 재검색 횟수에 따라 다음 Node를 결정한다."""
    # 충분하면 답변하고, 부족하면 한도 내 재검색 후 안전한 실패 응답으로 종료한다.
    if state["evidence_valid"]:
        return "generate_answer"
    if state["retry_count"] < MAX_RETRY:
        return "retry"
    return "insufficient_evidence"


assert basic_no_search["search_required"] is False
assert basic_no_search["search_queries"] == []
assert basic_search["search_required"] is True
assert len(basic_search["search_queries"]) == 1
assert len(basic_search["citations"]) > 0

assert route_after_evidence({"evidence_valid": True, "retry_count": 0}) == "generate_answer"
assert route_after_evidence({"evidence_valid": False, "retry_count": 0}) == "retry"
assert route_after_evidence({"evidence_valid": False, "retry_count": MAX_RETRY}) == "insufficient_evidence"
print("route_after_evidence 동작 확인 완료")


### 8.3 답변 생성과 근거 부족 응답 Node

근거가 충분하면 검색된 문서를 인용해 답변을 만들고 출처(`citations`)를
남긴다. 근거가 끝내 부족하면, 일반지식으로 확정적인 답변을 만드는 대신 근거
부족을 명시적으로 알린다.


In [ ]:
INSUFFICIENT_EVIDENCE_MESSAGE = "확인된 근거가 부족하여 확정적으로 답변할 수 없습니다. 관련 문서를 다시 확인해 주세요."


def generate_answer_node(state: RAGState) -> dict:
    # 최종 답변 프롬프트에는 검증을 통과한 검색 문서만 근거로 제공한다.
    docs_text = "\n".join(f"[{d['doc_id']}] {d['text']}" for d in state["retrieved_documents"])
    prompt = (
        f"질문: {state['question']}\n"
        "아래 근거 문서에 있는 내용만 사용해 답변하세요. 근거에 없는 내용은 추정하지 마세요.\n"
        f"{docs_text}"
    )
    answer = call_llm(prompt)
    citations = [{"doc_id": d["doc_id"], "section": d["section"]} for d in state["retrieved_documents"]]
    return {"final_answer": answer, "citations": citations}


def insufficient_evidence_node(state: RAGState) -> dict:
    # 금지 사항: 검색 결과가 없는데 LLM의 일반지식만으로 확정적인 답변을 생성하지 않는다.
    return {"final_answer": INSUFFICIENT_EVIDENCE_MESSAGE, "citations": []}


### 8.4 03-3B Graph 구성과 실행


In [ ]:
graph = StateGraph(RAGState)
graph.add_node("decide_search", decide_search_node)
graph.add_node("direct_answer", direct_answer_node)
graph.add_node("generate_query", generate_query_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("evaluate_evidence", evaluate_evidence_node)
graph.add_node("generate_answer", generate_answer_node)
graph.add_node("insufficient_evidence", insufficient_evidence_node)

graph.add_edge(START, "decide_search")
graph.add_conditional_edges(
    "decide_search",
    route_after_search_decision,
    {"direct_answer": "direct_answer", "generate_query": "generate_query"},
)
graph.add_edge("direct_answer", END)
graph.add_edge("generate_query", "retrieve")
graph.add_edge("retrieve", "evaluate_evidence")
# 근거가 부족하면 generate_query로 되돌아가 새 검색어로 다시 시도한다.
graph.add_conditional_edges(
    "evaluate_evidence",
    route_after_evidence,
    {
        "generate_answer": "generate_answer",
        "retry": "generate_query",
        "insufficient_evidence": "insufficient_evidence",
    },
)
graph.add_edge("generate_answer", END)
graph.add_edge("insufficient_evidence", END)

rag_app = graph.compile()


In [ ]:
# 그래프 그리기

from IPython.display import Image, display

display(
    Image(
        rag_app.get_graph().draw_mermaid_png()
    )
)


In [ ]:
# 시스템 테스트용 함수
def run_rag(question: str) -> dict:
    # 매 호출마다 초기 State를 새로 만들어 Graph 실행 간 상태를 격리한다.
    return rag_app.invoke(initial_rag_state(question))


## 9. B 실행 결과 관찰


In [ ]:
QUESTION_NO_SEARCH = "1 더하기 1은 얼마인가요?"
QUESTION_SUFFICIENT_FIRST_TRY = "생성형 인공지능 결과물 표시 의무는 무엇인가요?"
QUESTION_RETRY_THEN_SUFFICIENT = "고영향 인공지능 사업자가 이행해야 하는 조치는 무엇인가요?"
QUESTION_ALWAYS_INSUFFICIENT = "2027년 인공지능 관련 과태료 징수 실적은 얼마인가요?"

baseline_result = run_rag(QUESTION_SUFFICIENT_FIRST_TRY)
print(f"질문: {QUESTION_SUFFICIENT_FIRST_TRY}")
print(f"search_required: {baseline_result['search_required']}")
print(f"search_queries: {baseline_result['search_queries']}")
print(f"retry_count: {baseline_result['retry_count']}")
print(f"citations: {baseline_result['citations']}")
print(f"final_answer: {baseline_result['final_answer']}")


**결과 해석**: 조직 내부 지식 가드레일에 따라 `search_required=True`가 되어 검색 경로를 탔고, 첫 검색만으로
근거가 충분하다고 판단되어 `retry_count=0`으로 끝났다. `citations`에는 실제로
근거로 사용한 문서의 `doc_id`와 `section`이 남는다.


## 10. A와 B 비교 실험

### 10.1 일반 RAG vs Agentic RAG


In [ ]:
def naive_rag_answer(question: str) -> str:
    """일반 RAG: 항상 검색하고, 근거 적합성을 평가하지 않고 그대로 답변한다."""
    top_docs = retrieve(question, k=1)
    docs_text = "\n".join(f"[{d['doc_id']}] {d['text']}" for d in top_docs)
    prompt = f"질문: {question}\n근거 문서를 참고하여 답변을 작성하세요.\n{docs_text}"
    return call_llm(prompt)


naive_answer = naive_rag_answer(QUESTION_ALWAYS_INSUFFICIENT)
agentic_result = run_rag(QUESTION_ALWAYS_INSUFFICIENT)

print(f"질문: {QUESTION_ALWAYS_INSUFFICIENT}")
print(f"[일반 RAG] {naive_answer}")
print(f"[Agentic RAG] {agentic_result['final_answer']}")


**관찰**: 근거 문서에 답이 없을 때 일반 RAG가 항상 확정적인 오답을 만드는 것은
아니다. 이번 실행에서는 모델이 마침 "근거 문서에 2027년 과태료 징수 실적 정보가 없다"고
정직하게 답했다. 그런데 이 정직함은 프롬프트가 보장한 것이 아니라 모델이 그때
그때 판단한 결과일 뿐이다. 같은 프롬프트, 같은 상황에서도 모델이 무관한 문서를
근거로 그럴듯한 답을 만들어낼 위험은 여전히 남아 있으며, 실행할 때마다 결과가
달라질 수 있다는 것 자체가 문제다. Agentic RAG는 이 불확실성에 기대지 않고,
근거 적합성 평가에서 반복적으로 "부족"으로 판단하면 최대 재검색 횟수를 넘긴
뒤 근거 부족을 명시적으로 알린다. 모델의 선의를 매번 기대하는 대신, 검증 단계를
Workflow에 못 박아 두는 것이 핵심이다.


### 10.2 검색 불필요 경로 vs 검색 필요 경로


In [ ]:
no_search_result = run_rag(QUESTION_NO_SEARCH)
search_result = run_rag(QUESTION_SUFFICIENT_FIRST_TRY)

print(f"'{QUESTION_NO_SEARCH}'")
print(f"  search_required={no_search_result['search_required']}, search_queries={no_search_result['search_queries']}, citations={no_search_result['citations']}")
print(f"'{QUESTION_SUFFICIENT_FIRST_TRY}'")
print(f"  search_required={search_result['search_required']}, search_queries={search_result['search_queries']}, citations={len(search_result['citations'])}개")


**관찰**: 검색이 불필요하다고 판단된 질문은 `search_queries`가 비어 있고
`citations`도 없다. 검색이 필요한 질문만 실제 문서 검색과 출처 표시로
이어진다.


### 10.3 첫 검색 성공 vs 최대 재검색 소진


In [ ]:
first_try_result = run_rag(QUESTION_SUFFICIENT_FIRST_TRY)
retry_exhausted = run_rag(QUESTION_ALWAYS_INSUFFICIENT)

print(f"'{QUESTION_SUFFICIENT_FIRST_TRY}'")
print(f"  retry_count={first_try_result['retry_count']}, evidence_valid={first_try_result['evidence_valid']}")
print(f"'{QUESTION_ALWAYS_INSUFFICIENT}'")
print(f"  retry_count={retry_exhausted['retry_count']}, final_answer={retry_exhausted['final_answer']}")


**결과 해석**: 내부 지식 질문은 검색 경로에 진입한다. 첫 검색에서 충분하다고
판단될 수도 있고 재검색이 발생할 수도 있으므로, 특정 `retry_count`를 정답으로 보지 않는다.
대신 근거가 불충분한 상태에서는 인용이 붙은 확정 답변을 반환하지 않는 안전 불변조건을
검증한다.


### 10.4 LLM 호출 횟수 비교(비용과 지연시간)


In [ ]:
LLM_CALL_COUNT = 0
naive_rag_answer(QUESTION_ALWAYS_INSUFFICIENT)
naive_call_count = LLM_CALL_COUNT

LLM_CALL_COUNT = 0
run_rag(QUESTION_ALWAYS_INSUFFICIENT)
agentic_call_count = LLM_CALL_COUNT

print(f"같은 질문('{QUESTION_ALWAYS_INSUFFICIENT}')에 대한 LLM 호출 횟수")
print(f"  일반 RAG: {naive_call_count}회")
print(f"  Agentic RAG: {agentic_call_count}회")


**관찰**: 일반 RAG는 답변 생성 1회 호출로 끝나지만, Agentic RAG는 검색
필요성 판단, (재검색마다) 검색어 생성과 근거 적합성 평가가 반복되어 호출
횟수가 훨씬 많다. 이것이 3.9에서 언급한 "Agentic RAG가 복잡성, 호출 비용,
지연시간을 늘릴 수 있다"는 개념의 실측 결과다.


## 11. 실패 실험과 교정: 근거 없이 확정적으로 답변하기

작업지시서의 금지 사항을 직접 위반하는 코드를 만들어, 왜 위험한지 확인한다.


In [ ]:
def bad_naive_answer_always_confident(question: str) -> str:
    """금지된 패턴: 검색 결과가 실제로 근거가 되는지 확인하지 않고 항상 확정적으로 답한다."""
    top_docs = retrieve(question, k=1)
    docs_text = "\n".join(d["text"] for d in top_docs)
    prompt = f"질문: {question}\n근거 문서를 참고하여 답변을 작성하세요.\n{docs_text}"
    return call_llm(prompt)


bad_answer = bad_naive_answer_always_confident(QUESTION_ALWAYS_INSUFFICIENT)
good_result = run_rag(QUESTION_ALWAYS_INSUFFICIENT)

print("[금지된 방식] 근거 적합성 확인 없이 바로 답변:")
print(f"  {bad_answer}")
print("[올바른 방식] Agentic RAG의 근거 적합성 평가를 거친 응답:")
print(f"  {good_result['final_answer']}")

assert good_result["final_answer"] == INSUFFICIENT_EVIDENCE_MESSAGE
assert good_result["citations"] == []


**교정**: `bad_naive_answer_always_confident()`는 검색된 문서가 실제로
질문과 관련 있는지 확인하는 절차가 없다. 이번 실행에서는 모델이 우연히
정직하게 답했지만, 이 함수에는 그런 정직함을 보장하는 장치가 전혀 없다.
검증 절차가 없는 채로 반복 사용하면 언젠가 무관한 문서를 근거로 확정적인
답변을 만들어낼 위험이 항상 남아 있다. 올바른 방식은 `evaluate_evidence_node`처럼
근거 적합성을 명시적으로 평가하고, 부족하면 `insufficient_evidence_node`처럼 근거
부족을 알리는 것이다. 이는 모델이 이번에 정직했는지와 무관하게, 사용자가 잘못된
답변을 사실로 오인하는 것을 구조적으로 막는다.


## 12. 도전 과제

1. `MAX_RETRY` 값을 1로 줄이거나 4로 늘렸을 때, `QUESTION_RETRY_THEN_SUFFICIENT`와
   `QUESTION_ALWAYS_INSUFFICIENT`의 결과가 어떻게 달라지는지 관찰한다.
2. `generate_answer_node`의 프롬프트에 "근거 문서에 없는 내용은 답변에 포함하지
   마세요"라는 지시를 추가하고, 그 전후로 응답이 달라지는지 비교한다.
3. `citations`를 사람이 읽기 좋은 형태(예: `"[d3] 만족도 조사 결과"`)의 문자열
   목록으로 변환하는 함수를 추가로 작성한다.
4. `naive_rag_answer()`와 `bad_naive_answer_always_confident()`에 다른 질문을
   넣어보며, 모델이 실제로 무관한 문서를 근거로 확정적인 오답을 만들어내는
   질문을 직접 찾아본다. 어떤 질문 유형에서 환각이 더 잘 재현되는지 정리한다.


## 13. 테스트

**테스트 유형: 혼합 테스트 — 결정적 단위 테스트 + 외부 API 통합 테스트**

분기 함수는 결정적으로 검증하고, 전체 RAG 경로는 LLM·Embedding·Chroma를 함께 검증한다. 모델 판단에 따라 결과가 달라질 수 있으므로 실패하면 API·네트워크·모델 응답과 Collection 상태를 함께 확인한다.


In [ ]:
assert route_after_evidence({"evidence_valid": True, "retry_count": 0}) == "generate_answer"
assert route_after_evidence({"evidence_valid": False, "retry_count": 0}) == "retry"
assert route_after_evidence({"evidence_valid": False, "retry_count": MAX_RETRY}) == "insufficient_evidence"

assert decide_search_required(QUESTION_NO_SEARCH) is False
assert decide_search_required(QUESTION_SUFFICIENT_FIRST_TRY) is True
assert decide_search_required(QUESTION_ALWAYS_INSUFFICIENT) is True

no_search_check = run_rag(QUESTION_NO_SEARCH)
assert no_search_check["search_required"] is False
assert no_search_check["search_queries"] == []
assert no_search_check["citations"] == []

sufficient_check = run_rag(QUESTION_SUFFICIENT_FIRST_TRY)
assert sufficient_check["search_required"] is True
assert len(sufficient_check["search_queries"]) >= 1
if sufficient_check["evidence_valid"]:
    assert len(sufficient_check["citations"]) > 0
else:
    assert sufficient_check["final_answer"] == INSUFFICIENT_EVIDENCE_MESSAGE

exhausted_check = run_rag(QUESTION_ALWAYS_INSUFFICIENT)
assert exhausted_check["search_required"] is True
assert exhausted_check["retry_count"] <= MAX_RETRY
if not exhausted_check["evidence_valid"]:
    assert exhausted_check["final_answer"] == INSUFFICIENT_EVIDENCE_MESSAGE
    assert exhausted_check["citations"] == []

rag_state_fields = {
    "question", "search_required", "search_queries", "retrieved_documents",
    "evidence_valid", "retry_count", "citations", "final_answer",
}
assert set(RAGState.__annotations__.keys()) == rag_state_fields
print("테스트 통과")


## 14. 결과 저장


In [ ]:
agentic_rag_log = {
    "source_file": SOURCE_FILE,
    "law_name": LAW_NAME,
    "collection": COLLECTION_NAME,
    "max_retry": MAX_RETRY,
    "scenarios": [
        {"label": "search_not_required", "question": QUESTION_NO_SEARCH, "result": no_search_check},
        {"label": "sufficient_first_try", "question": QUESTION_SUFFICIENT_FIRST_TRY, "result": sufficient_check},
        {"label": "retry_exhausted", "question": QUESTION_ALWAYS_INSUFFICIENT, "result": exhausted_check},
    ],
    "llm_call_count_comparison": {
        "naive_rag": naive_call_count,
        "agentic_rag": agentic_call_count,
    },
}
saved_path = save_log(agentic_rag_log, OUTPUT_DIR / "logs" / "aitrust_3_agentic_rag_log.json")
print("저장 위치:", saved_path)


## 15. 핵심 정리

- A는 검색 필요 판단 → 최초 검색 → 답변의 한 방향 Graph를 먼저 완성한다.
- B는 A에 Query Rewrite, Evidence Validation, 재검색 Loop와 종료 조건을 추가한다.
- Agentic RAG는 검색 필요성 판단(`decide_search_required`), 검색어 생성
  (`generate_query_node`), 근거 적합성 평가(`evaluate_evidence_node`)를
  Workflow에 내장해, 일반 RAG보다 더 신중하게 검색과 답변을 수행한다.
- 재검색은 `route_after_evidence()`가 `evidence_valid`와 `retry_count`,
  `MAX_RETRY`를 함께 판단해 조건부로만 수행된다.
- 최대 재검색 횟수를 넘기면 근거 부족을 명시적으로 알리며, 일반지식만으로
  확정적인 답변을 만들지 않는다.
- Agentic RAG는 일반 RAG보다 LLM 호출 횟수가 많아 복잡성과 비용, 지연시간이
  늘어나므로, 항상 우수한 선택은 아니다.


## 16. 확인 문제

1. A 기본 Graph가 완성되었다고 판단할 수 있는 두 실행 경로는 무엇인가?
2. B가 A에 추가하는 판단과 Loop는 무엇인가?
3. `route_after_evidence()`가 `"retry"`와 `"insufficient_evidence"`를 구분하는
   기준은 무엇인가?
4. 검색 결과가 있는데도 근거 부족 응답을 해야 하는 경우는 언제인가?
5. Agentic RAG가 일반 RAG보다 항상 우수하지 않은 이유를 호출 횟수 관점에서
   설명하라.
